# Exercise 2: Creating MongoDB Collections and NoSQL Data Modeling

In Exercise 1, we compared MongoDB to PostgreSQL and saw the trade-offs. Now it's time to build something from scratch!

In this exercise, you'll learn:
1. **When** to use a document store (use cases and application maturity)
2. **How** to create collections and insert documents in MongoDB
3. **Data modeling principles** for NoSQL: embedding data, avoiding joins, and leveraging schema flexibility

## The Scenario: Building a Blog Platform

Imagine you're building a modern blog platform. Your application is growing, and you need a database that can:
- Handle rapidly changing content structures (some posts have videos, some have polls, some have code snippets)
- Store nested data naturally (posts with comments, comments with replies)
- Scale horizontally as traffic grows
- Support fast reads for displaying complete blog posts

This is a **perfect use case for MongoDB!**

## Step 1: Setting Up MongoDB Connection

Let's connect to MongoDB using PyMongo.

(Run this code block to set up the connection)

In [ ]:
%%capture
%pip install pymongo pandas;
import pymongo
from pymongo import MongoClient
import pandas as pd
from datetime import datetime

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')

# Create/access a database for our blog
db = client['blog_platform']

print("✅ Connected to MongoDB")
print(f"Database: {db.name}")
print(f"Available collections: {db.list_collection_names()}")

## Step 2: Creating Your First Collection

In MongoDB, collections are created automatically when you insert the first document. No need for CREATE TABLE statements!

Let's create a collection for blog posts.

### Data Modeling Principle #1: Embed Related Data

Instead of having separate tables for posts and authors (and requiring JOINs), we **embed** author information directly in each post.

In [ ]:
# Access (and create if not exists) the posts collection
posts = db.posts

# Clear any existing data for this exercise
posts.drop()

# Insert our first blog post with embedded author data
first_post = {
    "title": "Getting Started with MongoDB",
    "content": "MongoDB is a document-oriented database that makes storing complex data simple...",
    "author": {                          # Embedded document!
        "name": "Alice Smith",
        "email": "alice@example.com",
        "bio": "Tech writer and database enthusiast"
    },
    "tags": ["mongodb", "databases", "nosql"],  # Arrays are native!
    "published_date": datetime(2024, 1, 15),
    "views": 0,
    "status": "published"
}

# Insert the document
result = posts.insert_one(first_post)
print(f"✅ Inserted post with ID: {result.inserted_id}")

# Verify the insert
inserted_post = posts.find_one({"_id": result.inserted_id})
print(f"\nInserted document:")
pd.DataFrame([inserted_post])

""


**Key Points:**
- ✅ No schema definition needed
- ✅ Author data is embedded (no JOIN required to display the post)
- ✅ Arrays (`tags`) are stored natively
- ✅ MongoDB auto-generates `_id` as the primary key

## Step 3: Inserting Multiple Documents

Let's add more posts. Notice how we can insert documents with **different structures** in the same collection!

### Data Modeling Principle #2: Schema Flexibility

(Fill in the blanks to complete the second post)

In [ ]:
# Insert multiple posts with varying structures
more_posts = [
    {
        "title": "Python Best Practices",
        "content": "Writing clean, maintainable Python code requires discipline...",
        "author": {
            "name": "Bob Johnson",
            "email": "bob@example.com"
            # Note: Bob doesn't have a bio field
        },
        "tags": ["python", "programming"],
        "published_date": datetime(2024, 1, 16),
        "views": 0,
        "status": "published",
        "featured": True  # New field! Not in the first post
    },
    {
        "title": "Video Tutorial: Docker Basics",
        "content": "Learn Docker fundamentals in this comprehensive video guide...",
        "author": {
            "name": "Alice Smith",  # ??? - Author name
            "email": "alice@example.com"
        },
        "tags": ["docker", "devops", "tutorial"],
        "published_date": datetime(2024, 1, 17),
        "views": 0,
        "status": "published",
        "media": {  # This post has media!
            "type": "video",
            "url": "https://example.com/docker-tutorial.mp4",
            "duration_minutes": 45
        }
    },
    {
        "title": "Draft: Database Comparison",
        "content": "Comparing SQL and NoSQL databases...",
        "author": {
            "name": "Charlie Day",
            "email": "charlie@example.com",
            "bio": "Database architect"
        },
        "tags": ["databases"],
        "created_date": datetime(2024, 1, 18),  # Different date field for drafts
        "status": "draft"  # This post is not published yet
        # Note: No published_date or views for drafts
    }
]

result = posts.insert_many(more_posts)
print(f"✅ Inserted {len(result.inserted_ids)} posts")
print(f"Total posts in collection: {posts.count_documents({})}")

""


Notice:
- Each document can have **different fields**
- Some posts have `media`, some have `featured`, some have neither
- Drafts use `created_date` instead of `published_date`
- No schema changes or migrations needed!

## Step 4: Adding Nested Comments

Now let's add comments to a post. This demonstrates **deep embedding** - comments with replies.

### Data Modeling Principle #3: Embed Nested Hierarchies

(Fill in the blanks to complete the nested structure)

In [ ]:
# Find the first post and add comments to it
first_post_id = posts.find_one({"title": "Getting Started with MongoDB"})["_id"]

# Update the post to add an embedded comments array
comments_data = {
    "comments": [
        {
            "comment_id": 1,
            "author": "Diana Prince",
            "text": "Great introduction! Very helpful.",
            "date": datetime(2024, 1, 15, 10, 30),
            "likes": 5,
            "replies": [  # Nested replies!
                {
                    "reply_id": 1,
                    "author": "Alice Smith",  # ??? - Post author responding
                    "text": "Thanks! Glad you found it useful.",
                    "date": datetime(2024, 1, 15, 11, 0)
                }
            ]
        },
        {
            "comment_id": 2,
            "author": "Eve Davis",
            "text": "Could you cover MongoDB aggregation in the next post?",
            "date": datetime(2024, 1, 15, 14, 20),
            "likes": 3,
            "replies": []  # No replies yet
        }
    ]
}

# Update the post
posts.update_one(
    {"_id": first_post_id},
    {"$set": comments_data}
)

print("✅ Added comments to the post")

# Retrieve and display the post with comments
post_with_comments = posts.find_one({"_id": first_post_id})
print("\nPost with embedded comments:")
print(f"Title: {post_with_comments['title']}")
print(f"Number of comments: {len(post_with_comments['comments'])}")
print(f"\nFirst comment: {post_with_comments['comments'][0]['text']}")
print(f"Replies: {len(post_with_comments['comments'][0]['replies'])}")

""


**Why this is powerful:**
- ✅ Entire post with all comments loaded in **one query**
- ✅ No JOINs needed across multiple tables
- ✅ Natural hierarchy: comments → replies → nested replies
- ✅ Perfect for displaying a blog post with all its content

## Step 5: Querying the Collection

Let's query our posts using MongoDB's flexible query syntax.

### Find all published posts

# Query for published posts only
published_posts = list(posts.find({"status": "published"}))

print(f"Found {len(published_posts)} published posts:\n")
for post in published_posts:
    print(f"- {post['title']} by {post['author']['name']}")

In [ ]:
### Find posts by a specific author

(Fill in the blank to query posts by Alice Smith)

RuntimeError: (psycopg2.errors.NotNullViolation) null value in column "email" of relation "customers" violates not-null constraint
DETAIL:  Failing row contains (3, Charlie Brown, null).

[SQL: INSERT INTO Customers (customer_id, full_name)
VALUES (3, 'Charlie Brown');]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


# Query nested fields using dot notation
alice_posts = list(posts.find({"author.name": "Alice Smith"}))  # ??? - Use dot notation for nested fields

print(f"Alice has written {len(alice_posts)} posts:\n")
for post in alice_posts:
    print(f"- {post['title']}")

In [ ]:
### Find posts with specific tags

MongoDB has powerful operators for querying arrays.

""


# Find posts tagged with "mongodb"
mongodb_posts = list(posts.find({"tags": "mongodb"}))  # Array contains this value

print(f"Posts tagged with 'mongodb':")
for post in mongodb_posts:
    print(f"- {post['title']}")
    print(f"  Tags: {', '.join(post['tags'])}")

In [ ]:
## Step 6: Updating Documents

MongoDB provides powerful update operators. Let's update some posts.

### Increment view counts

Every time someone reads a post, we increment the view count.

RuntimeError: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "customers_email_key"
DETAIL:  Key (email)=(alice@example.com) already exists.

[SQL: INSERT INTO Customers (customer_id, full_name, email)
VALUES (4, 'Eve Davis', 'alice@example.com');]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


# Simulate viewing the first post 10 times
post_to_view = posts.find_one({"title": "Getting Started with MongoDB"})

# Use $inc to increment the views field
posts.update_one(
    {"_id": post_to_view["_id"]},
    {"$inc": {"views": 10}}  # Increment by 10
)

print("✅ Updated view count")

# Verify
updated_post = posts.find_one({"_id": post_to_view["_id"]})
print(f"Views: {updated_post['views']}")

In [ ]:
### Add a new tag to a post

Let's add a tag to the Python post using the `$push` operator.

(Fill in the blank to push a new tag)

""


# Add a new tag to the Python post
posts.update_one(
    {"title": "Python Best Practices"},
    {"$push": {"tags": "best-practices"}}  # ??? - Use $push to add to array
)

print("✅ Added new tag")

# Verify
python_post = posts.find_one({"title": "Python Best Practices"})
print(f"Updated tags: {python_post['tags']}")

In [ ]:
### Add a new comment to a post

Let's add another comment to our first post.

RuntimeError: (psycopg2.errors.CheckViolation) new row for relation "accounts" violates check constraint "accounts_balance_check"
DETAIL:  Failing row contains (101, 1, -50.00).

[SQL: UPDATE Accounts
SET balance = -50.00
WHERE account_id = 101;]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


# Add a new comment
new_comment = {
    "comment_id": 3,
    "author": "Frank Miller",
    "text": "Looking forward to more posts like this!",
    "date": datetime(2024, 1, 18, 9, 0),
    "likes": 0,
    "replies": []
}

posts.update_one(
    {"title": "Getting Started with MongoDB"},
    {"$push": {"comments": new_comment}}
)

print("✅ Added new comment")

# Verify
post = posts.find_one({"title": "Getting Started with MongoDB"})
print(f"Total comments: {len(post['comments'])}")
print(f"Latest comment: {post['comments'][-1]['text']}")

In [ ]:
## Step 7: When to Embed vs. Reference

So far, we've embedded everything. But should you **always** embed? Let's explore when to use references instead.

### Data Modeling Principle #4: Reference When Data is Shared or Large

Let's create a **users** collection and reference it from posts instead of embedding.

**Why reference users?**
- User data is shared across many posts
- If a user updates their profile, we don't want to update hundreds of posts
- Users have complex data (preferences, activity history, etc.)

""


# Create a users collection
users = db.users
users.drop()

# Insert users
user_docs = [
    {
        "user_id": 1,
        "name": "Alice Smith",
        "email": "alice@example.com",
        "bio": "Tech writer and database enthusiast",
        "registration_date": datetime(2024, 1, 1),
        "preferences": {
            "theme": "dark",
            "notifications": True
        }
    },
    {
        "user_id": 2,
        "name": "Bob Johnson",
        "email": "bob@example.com",
        "registration_date": datetime(2024, 1, 5),
        "preferences": {
            "theme": "light",
            "notifications": False
        }
    }
]

users.insert_many(user_docs)
print(f"✅ Created {users.count_documents({})} users")

# Now create a post that REFERENCES a user instead of embedding
referenced_post = {
    "title": "MongoDB vs PostgreSQL: When to Use Each",
    "content": "Choosing the right database depends on your use case...",
    "author_id": 1,  # Reference to user_id instead of embedding!
    "tags": ["databases", "comparison"],
    "published_date": datetime(2024, 1, 19),
    "views": 0,
    "status": "published"
}

posts.insert_one(referenced_post)
print("✅ Created post with user reference")

In [ ]:
To display this post with author info, we need to fetch it in two steps (application-level join):

RuntimeError: (psycopg2.errors.ForeignKeyViolation) insert or update on table "accounts" violates foreign key constraint "fk_customer"
DETAIL:  Key (customer_id)=(999) is not present in table "customers".

[SQL: INSERT INTO Accounts (account_id, customer_id, balance)
VALUES (103, 999, 50.00);]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


# Fetch the post
post = posts.find_one({"title": "MongoDB vs PostgreSQL: When to Use Each"})

# Fetch the author separately
author = users.find_one({"user_id": post["author_id"]})

print(f"Post: {post['title']}")
print(f"Author: {author['name']}")
print(f"Author email: {author['email']}")
print(f"Author preferences: {author['preferences']}")

## Step 8: Data Modeling Decision Guide

Here's when to use each approach:

### ✅ Embed Data When:
- Data is **frequently read together** (posts + comments)
- Child data **belongs to** one parent (order items belong to one order)
- Data has a **one-to-few** relationship (a post has 5-50 comments)
- Child data is **not updated often**
- You want **fast reads** (one query gets everything)

### ✅ Reference Data When:
- Data is **shared** across documents (users write many posts)
- Child data is **updated frequently** (user profiles)
- Data has a **one-to-many** relationship (a user has 1000+ posts)
- Child data is **large** (would make parent document huge)
- You need to query child data **independently**

### ⚖️ Hybrid Approach (Best Practice):
- Store minimal reference data in parent (user_id + name)
- Keep full data in referenced collection
- Update denormalized data only when critical fields change

In [ ]:
## Step 9: Practical Exercise - Create Your Own Collection

Now it's your turn! Create a collection for an e-commerce product catalog.

**Requirements:**
1. Create a `products` collection
2. Each product should have:
   - Product name and description
   - Price
   - Category
   - Embedded specifications (varying by category)
   - Array of customer reviews (with nested replies)
3. Insert at least 3 products with different structures

(Fill in the code below)

c:\Users\ktoto\AppData\Local\Programs\Python\Python313\Lib\site-packages\sql\connection\connection.py:881: JupySQLRollbackPerformed: Current transaction is aborted. JupySQL executed a ROLLBACK operation.
  warnings.warn(
RuntimeError: (psycopg2.errors.InFailedSqlTransaction) current transaction is aborted, commands ignored until end of transaction block

[SQL: SELECT * FROM Customers;]
(Background on this error at: https://sqlalche.me/e/20/2j85)


# Create products collection
products = db.products
products.drop()

# Your code here: Insert 3 products
# Example structure:
product_docs = [
    {
        "product_id": 1,
        "name": "Gaming Laptop",  # ??? - Product name
        "description": "High-performance laptop for gaming and creative work",
        "price": 1299.99,
        "category": "Electronics",
        "specs": {  # Embedded specifications
            "processor": "Intel i7",
            "ram": "16GB",
            "storage": "512GB SSD",
            "screen": "15.6 inch"
        },
        "reviews": [
            {
                "reviewer": "John Doe",
                "rating": 5,
                "comment": "Excellent laptop!",
                "date": datetime(2024, 1, 10)
            }
        ]
    },
    # ??? - Add 2 more products with different specs
]

result = products.insert_many(product_docs)
print(f"✅ Inserted {len(result.inserted_ids)} products")

# Verify
all_products = list(products.find())
for prod in all_products:
    print(f"\n{prod['name']}: ${prod['price']}")
    print(f"  Category: {prod['category']}")
    if 'reviews' in prod:
        print(f"  Reviews: {len(prod['reviews'])}")

In [ ]:
## Key Takeaways: MongoDB Data Modeling Principles

**1. Think in Documents, Not Tables**
- Model data the way your application uses it
- One query should fetch what one screen needs

**2. Embed for Performance**
- Embed data that's read together
- Reduces round trips and JOINs
- Perfect for one-to-few relationships

**3. Reference for Consistency**
- Reference data that's shared or updated often
- Prevents data duplication issues
- Good for one-to-many or many-to-many

**4. Leverage Schema Flexibility**
- Different documents can have different fields
- No migrations needed for new features
- Great for evolving applications

**5. Optimize for Your Queries**
- Design based on read/write patterns
- Denormalize for reads, normalize for writes
- There's no "perfect" schema—it depends on your use case

**6. Document Size Limits**
- MongoDB documents limited to 16MB
- If you're hitting this limit, use references
- Split large arrays into separate collections

MongoDB's official guide: https://www.mongodb.com/docs/manual/data-modeling/

""


In [ ]:
%%sql
BEGIN;
INSERT INTO Orders (order_id, customer_id, product, quantity, total, order_date)
VALUES (107, 3, 'Headphones', 'one', 'ninety-nine', '2024-01-21');
COMMIT;

,customer_id,full_name,email
0,1,Alice Smith,alice@example.com
1,2,Bob Johnson,bob@example.com


🛑 **PostgreSQL rejects this with a type error!** The schema enforces data integrity.

## Step 8: When to Use MongoDB vs PostgreSQL

### Use MongoDB When:
✅ Schema changes frequently and unpredictably
✅ Data is naturally nested (e.g., blog posts with comments)
✅ You need horizontal scaling across many servers
✅ Most queries read complete documents (no complex joins)
✅ You need fast writes and reads for simple queries

### Use PostgreSQL When:
✅ Data relationships are complex and require many joins
✅ Data integrity and consistency are critical (financial data, etc.)
✅ Schema is well-defined and stable
✅ You need ACID transactions across multiple tables
✅ Business intelligence and precise calculations are required

### The Hybrid Approach:
Many modern applications use **both**:
- PostgreSQL for core transactional data (orders, payments, users)
- MongoDB for flexible data (product catalogs, user preferences, logs)

In [ ]:
## Key Takeaways

**MongoDB Strengths:**
- Fast reads for complete documents
- Flexible schemas that evolve without migrations
- Horizontal scaling for massive datasets
- Great for rapid prototyping

**MongoDB Weaknesses:**
- Joins ($lookup) are slow and inefficient
- No schema enforcement can lead to data quality issues
- Harder to ensure data consistency
- Not ideal for complex relational queries

**PostgreSQL Strengths:**
- Excellent JOIN performance
- Strong data consistency and type checking
- ACID transactions
- Perfect for complex queries and business intelligence

**PostgreSQL Weaknesses:**
- Schema changes require migrations
- Vertical scaling has limits
- Less flexible for rapidly evolving data models

**The Bottom Line:** Choose your database based on your data's **nature** and **access patterns**, not trends or popularity.

✅ Transaction committed: Transfer successful


Expected Result: ✅ Success. Alice will have 900.00 dollars and Bob will have 350.00. Both updates worked, so the COMMIT made them permanent.

Let's move onto the "Failure Path" i.e. an Atomic Rollback. Start with a transfer $5,000 from Alice to Bob. Alice only has $900. The CHECK constraint will stop this.

#### Task 5: Set the addition and subtraction values of the balance query to balance - 5000 and balance + 5000.

In [ ]:
conn = get_connection()
cursor = conn.cursor()

try:
    # Step 1: Subtract $5000 from Alice (This will fail the CHECK constraint)
    cursor.execute("UPDATE Accounts SET balance = balance - 5000.00 WHERE account_id = 101;")
    
    # Step 2: Add $5000 to Bob (This line may not even be reached)
    cursor.execute("UPDATE Accounts SET balance = balance + 5000.00 WHERE account_id = 102;")
    

    conn.commit()  # Try to finalize
    print("Transaction committed")
    
except Exception as e:
    conn.rollback()
    print(f"Transaction rolled back: {e}")
    
finally:
    cursor.close()
    conn.close()

❌ Transaction rolled back: new row for relation "accounts" violates check constraint "accounts_balance_check"
DETAIL:  Failing row contains (101, 1, -4100.00).



Expected Result: Failure. The database will throw a CHECK constraint error (from Exercise 1C) when it tries to execute the first UPDATE. The COMMIT will fail and/or you'll be forced to ROLLBACK. 
In general, we will ROLLBACK regardless, as it's good practice to do so after a transaction has failed. 

Let's see if this failed correctly - i.e. that both transactions failed, not just one. 

In [18]:
%%sql
SELECT * FROM Accounts;

,account_id,customer_id,balance
0,101,1,900.00
1,102,2,350.00


You will see that Alice still has 900.00 dollars and Bob still has 350.00. Crucially, Bob did not get $5000, and Alice's account was not set to a negative value. Because one part of the transaction failed (Step 1), the entire transaction was aborted. This is Atomicity.

### Part 3: Testing 'I' (Isolation)
We've shown the database can guarantee Atomicity, and Consistency. Next, we want to show that one transaction doesn't see the "dirty" or uncommitted work of another. This requires two separate database connections - which we can simulate by using multiple notebook cells with %sql magic, where each transaction is kept open until explicitly committed or rolled back.

Here's the starting Point: Alice (101) has $900.00.

In Session 1 (First Cell) we will:
1. Run BEGIN TRANSACTION;	
2. Run UPDATE Accounts SET balance = 50.00 WHERE account_id = 101; (Alice's new balance is $50, but don't COMMIT!)	
3. Check the balance: SELECT balance FROM Accounts WHERE account_id = 101; 

You should see the balance is 50.00 dollars within this transaction.

NEXT - 

4. In a separate cell (Session 2), open a new connection and check the balance: SELECT balance FROM Accounts WHERE account_id = 101;
Expected Result: Session 2 sees **900.00** dollars. It does not see the uncommitted 50.00 dollar value. It is isolated from Session 1's incomplete work.
5. Back in Session 1, run: COMMIT;	
6. Immediately after Session 1 commits, run this again in Session 2: SELECT balance FROM Accounts WHERE account_id = 101;
Expected Result: Now Session 2 sees **$50.00**. It can only see the new value after it has been fully and durably committed.

As you can see, isolation prevents other connections from reading "dirty" or "in-progress" data, which would lead to massive confusion and inconsistency.

In [19]:
# Session 1: Start a transaction and update Alice's balance (but don't commit yet)
conn1 = get_connection()
cursor1 = conn1.cursor()


# Begin transaction and update# Keep connection open for next cells (don't close yet!)

cursor1.execute("UPDATE Accounts SET balance = 50.00 WHERE account_id = 101;")
# Check the balance within this transactionresult = cursor1.fetchone()
cursor1.execute("SELECT balance FROM Accounts WHERE account_id = 101;")
rows = cursor1.fetchall()
print("Query results:")
for row in rows:
    print(row)
        

Query results:
(Decimal('50.00'),)


In [20]:
# Session 2: Open a separate connection to see if it can see uncommitted changes
conn2 = get_connection()
cursor2 = conn2.cursor()

cursor2.execute("SELECT balance FROM Accounts WHERE account_id = 101;")
rows = cursor2.fetchall()
print("Expected: $900.00 (Session 2 cannot see uncommitted changes from Session 1)")
print(f"Session 2 sees (before commit): ${rows[0][0]}")

Expected: $900.00 (Session 2 cannot see uncommitted changes from Session 1)
Session 2 sees (before commit): $900.00


#### Task 6: Commit the transaction using conn1.commit(). Close the connection and the cursor using the .close() functionality

In [ ]:
# Session 1: Now commit the transaction
conn1.commit()

print("Session 1 transaction committed")
conn1.close()
cursor1.close()

✅ Session 1 transaction committed


In [22]:
# Session 2: Open another connection to verify committed changes are now visible
conn2.close()
cursor2.close()

conn2 = get_connection()
cursor2 = conn2.cursor()

cursor2.execute("SELECT balance FROM Accounts WHERE account_id = 101;")
result = cursor2.fetchone()
print("Expected: $50.00 (Session 2 can now see committed changes from Session 1)")
print(f"Session 2 sees (after commit): ${result[0]}")

Expected: $50.00 (Session 2 can now see committed changes from Session 1)
Session 2 sees (after commit): $50.00


Which is exactly what we expected! The isolation property works perfectly.

So far, we have the ACI - Atomicity, Consistency, and Isolation gaurantees of the database tested. Let's add the D in ACID. 

### Part 4: Testing 'D' (Durability)
We need to show that once COMMIT is executed, the data is safe and permanent. In most terms, this usually means it's stored on disk and not volatile memory. Let's give Bob a $77.00 bonus. Before we do that, let's see what is currently in the accounts:


In [23]:
%%sql
SELECT * FROM Accounts
JOIN Customers ON Accounts.customer_id = Customers.customer_id;

,account_id,customer_id,balance,customer_id,full_name,email
0,102,2,350.00,2,Bob Johnson,bob@example.com
1,101,1,50.00,1,Alice Smith,alice@example.com


In [ ]:
conn = get_connection()
cursor = conn.cursor()

try:
    cursor.execute("UPDATE Accounts SET balance = balance + 77.00 WHERE account_id = 102;")    
    conn.commit()    
    
    print("Transaction committed: Bob received $77.00 bonus")

except Exception as e:    
    print(f"Transaction rolled back: {e}")
    conn.rollback()
    
finally:
    cursor.close()
    conn.close()

✅ Transaction committed: Bob received $77.00 bonus


Let's make sure the data is all there. We should see Bob's new balance, with his increased bonus.


In [25]:
%%sql
SELECT balance FROM Accounts WHERE account_id = 102;

,balance
0,427.00


Here comes the test: Now, we'll simulate a database restart by creating a fresh connection in the next cell. This demonstrates durability - that committed data survives even after connections are closed.

#### Task 7: Open another connection called new_conn, and set the cursor to new_conn.cursor().

In [ ]:
new_conn = get_connection()
cursor = new_conn.cursor()

try:
    cursor.execute("SELECT * FROM Accounts JOIN Customers ON Accounts.customer_id = Customers.customer_id WHERE account_id = 102;")    
    new_conn.commit()    
    
    print("Transaction is durable: Bob still has the same balance after reconnecting")

    result = cursor.fetchone()
    print(f"${result}")

except Exception as e:    
    print(f"Transaction rolled back: {e}")
    new_conn.rollback()
    
finally:
    cursor.close()
    new_conn.close()

✅ Transaction is durable: Bob still has the same balance after reconnecting
$(102, 2, Decimal('427.00'), 2, 'Bob Johnson', 'bob@example.com')


Lesson: The value is still the same. Because the transaction was COMMITted, the database guarantees that this change is Durable and has survived the "crash" (i.e., you disconnecting). This is the opposite of ROLLBACK.